In [20]:
import tifffile as tiff
from pathlib import Path
import pickle
import numpy as np

SCRATCH_DIR = Path("/root/capsule/scratch")
MORPH_DIR = SCRATCH_DIR / "morphology"
MORPH_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("/root/capsule/data")

In [8]:
data_info = pickle.load(open('/root/capsule/code/Preprocessing/data_info.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [778174, 786297, 797371]


In [36]:
mouse_id = mouse_ids[0]
zstack_name = list((DATA_DIR / data_info[mouse_id]['cortical_zstack']['segmented']).glob("*"))[0]
zstack_size_um = [int(i) for i in zstack_name.name.split('-')[3].split('x')[::-1]]
masks_dir = zstack_name / "segmentation_masks_original.tif"
masks = tiff.imread(masks_dir)
zstack_size_pixels = masks.shape
zstack_resolution = [zstack_size_um[i] / zstack_size_pixels[i] for i in range(3)]
zstack_resolution

[1.0, 1.3671875, 1.3671875]

In [40]:
# np is already imported in an earlier cell; avoid re-importing it here
import pandas as pd

for mouse_id in mouse_ids:
    output_path = MORPH_DIR / f"{mouse_id}_mask_properties.csv"
    if output_path.exists():    
        print(f"  → {output_path.name} already exists – skipping.")
        continue
    zstack_name = list((DATA_DIR / data_info[mouse_id]['cortical_zstack']['segmented']).glob("*"))[0]
    zstack_size_um = [int(i) for i in zstack_name.name.split('-')[3].split('x')[::-1]]
    masks_dir = zstack_name / "segmentation_masks_original.tif"
    masks = tiff.imread(masks_dir)
    zstack_size_pixels = masks.shape
    zstack_resolution = [zstack_size_um[i] / zstack_size_pixels[i] for i in range(3)]

    labels = np.unique(masks)
    labels = labels[labels != 0]

    res = np.array(zstack_resolution)  # [z_res, y_res, x_res]
    results = []

    for lab in labels:
        coords = np.column_stack(np.where(masks == lab))  # z,y,x (pixels)
        coords_um = coords * res  # convert to microns
        centroid = coords_um.mean(axis=0)  # z,y,x

        # size in original x,y,z (micron)
        size_z = np.ptp(coords_um[:, 0])
        size_y = np.ptp(coords_um[:, 1])
        size_x = np.ptp(coords_um[:, 2])

        # PCA to get principal axes
        coords_centered = coords_um - centroid
        if coords_centered.shape[0] >= 3:
            _, _, Vt = np.linalg.svd(coords_centered, full_matrices=False)
            pcs = Vt  # rows: principal axes in (z,y,x) coordinates
        else:
            pcs = np.eye(3)

        # sizes along principal axes (micron)
        proj = coords_centered @ pcs.T
        size_pc1 = np.ptp(proj[:, 0])
        size_pc2 = np.ptp(proj[:, 1])
        size_pc3 = np.ptp(proj[:, 2])

        # angle of dominant axis in XY plane (degrees)
        vx = pcs[0, 2]  # x component of first principal axis
        vy = pcs[0, 1]  # y component
        angle_deg_xy = float(np.degrees(np.arctan2(vy, vx)))

        results.append({
            "label": int(lab),
            "centroid_x_um": float(centroid[2]),
            "centroid_y_um": float(centroid[1]),
            "centroid_z_um": float(centroid[0]),
            "angle_deg_xy": angle_deg_xy,
            "size_x_um": float(size_x),
            "size_y_um": float(size_y),
            "size_z_um": float(size_z),
            "size_pc1_um": float(size_pc1),
            "size_pc2_um": float(size_pc2),
            "size_pc3_um": float(size_pc3),
            "n_voxels": int(coords.shape[0])
        })

    df_masks_props = pd.DataFrame(results).sort_values("label").reset_index(drop=True)
    print(df_masks_props)
    df_masks_props.to_csv(output_path, index=False)

  → 778174_mask_properties.csv already exists – skipping.
       label  centroid_x_um  centroid_y_um  centroid_z_um  angle_deg_xy  \
0          1     696.203954     452.455045      47.435754    -89.672838   
1          2     693.803089      28.642372      54.019057    -82.361345   
2          3     658.254736     269.229679      49.585492    101.658131   
3          4     562.129317     298.165159      56.932970    -17.986200   
4          5     510.335599     222.313890      57.276923    -65.212736   
...      ...            ...            ...            ...           ...   
19330  19331     423.363416     386.765894     446.763547    177.777251   
19331  19332     154.001510     469.226299     446.521472    -32.251497   
19332  19333     539.180596     613.755905     446.430233     -5.660369   
19333  19334      53.492106     259.457829     447.403141    173.202096   
19334  19335     207.709316     248.209021     447.798742    -38.725891   

       size_x_um  size_y_um  size_z_um  s

In [39]:
df_masks_props

,label,centroid_x_um,centroid_y_um,centroid_z_um,angle_deg_xy,size_x_um,size_y_um,size_z_um,size_pc1_um,size_pc2_um,size_pc3_um,n_voxels
0,1,692.174889,565.449736,19.875828,73.690777,9.570312,12.304688,13.0,15.991646,12.827907,11.014561,604
1,2,669.409502,627.508166,20.009416,137.148731,13.671875,13.671875,14.0,20.572950,13.890408,14.183134,531
2,3,640.005327,646.835050,19.381818,53.920736,9.570312,5.468750,11.0,11.913669,10.198415,6.054104,220
3,4,674.470403,652.904334,18.947115,22.065248,9.570312,6.835938,10.0,10.160326,10.771857,6.296226,208
4,5,539.065176,688.973539,20.623288,105.709807,8.203125,8.203125,16.0,16.147715,10.266319,8.437656,292
...,...,...,...,...,...,...,...,...,...,...,...,...
15146,15147,245.377604,593.337674,446.804233,11.820388,15.039062,8.203125,6.0,16.508274,8.217908,6.686526,189
15147,15148,458.733259,444.698661,446.108844,-22.583113,8.203125,9.570312,5.0,10.770837,9.109087,5.491024,147
15148,15149,351.623535,546.652832,447.350000,-97.639701,6.835938,9.570312,5.0,10.179814,8.064127,5.846557,160
15149,15150,288.134766,558.947266,447.520000,-81.317749,9.570312,10.937500,5.0,11.882539,9.715574,5.285849,200
